# ДЗ 2: первый простой Transformer

Первый пробный вариант: 512 пар из первого train-файла, 200 обновлений, 2 слоя энкодера и 2 декодера, `d_model=128`. Токенизация — UTF-8-байты; позиции обучаемые. Обычный FP32 и greedy-генерация.

В Colab выберите T4 и запустите ячейки по порядку. Нужны PyTorch и PyArrow; при отсутствии PyArrow выполните `%pip install pyarrow`. Код скачивает один train-файл и официальный test, сохраняет пробный checkpoint и результаты в текущем каталоге. Программы датасета и генерации не исполняются; ответы в бот не отправляются.

Ноутбук ещё не запускался на T4. Класс модели локально проверен на CPU; детали — в `first-baseline.md`. Максимум генерации 512 выходных токенов (байты и EOS): около 97,2% программ в исследованной выборке помещаются, более длинные ответы могут обрезаться. Генерации первого прогона нужно считать пробными.

Если сессия Colab прервётся, файлы виртуальной машины могут исчезнуть. Скачайте `hw02-first-baseline.pt` и `hw02-first-results.json` после завершения; сохранение checkpoint само по себе не копирует его в Drive. [FAQ Colab](https://research.google.com/colaboratory/faq.html).

In [ ]:
# HW2_SIMPLE_BASELINE: small first experiment, not a final submission.
import ast
import hashlib
import json
import math
import random
import time
from pathlib import Path
from urllib.request import urlretrieve

import torch
from torch import nn
from torch.nn.utils.rnn import pad_sequence
import pyarrow.parquet as pq

SEED, STEPS, BATCH_SIZE = 42, 200, 8
D_MODEL, N_LAYERS, MAX_POSITIONS = 128, 2, 2048
PAD, BOS, EOS, VOCAB = 0, 1, 2, 259  # 256 UTF-8 bytes + 3 special tokens
random.seed(SEED)
torch.manual_seed(SEED)
assert torch.cuda.is_available(), 'Enable GPU in Colab before running this notebook'
device = torch.device('cuda')
print('Device:', torch.cuda.get_device_name(0) if device.type == 'cuda' else 'CPU')
print('PyTorch:', torch.__version__)



In [ ]:
# 1. Data: one allowed shard, a small training subset, and question-level validation.
revision = 'b2724689ff079e9c22efd548fe134bfb29036dbb'
base_url = f'https://huggingface.co/datasets/SlavaYus/tinysms_qa_12m/resolve/{revision}'
cache = Path('hw02-data')
cache.mkdir(exist_ok=True)
for name, remote in [('train.parquet', 'data/train/part-00000.parquet'), ('test.parquet', 'data/test/test.parquet')]:
    if not (cache / name).exists():
        urlretrieve(f'{base_url}/{remote}', cache / name)
rows = next(pq.ParquetFile(cache / 'train.parquet').iter_batches(batch_size=4096)).to_pylist()
def is_validation(question):
    key = hashlib.sha256(f'{SEED}:{question}'.encode()).digest()
    return int.from_bytes(key[:8], 'big') % 10 == 0
short_rows = [r for r in rows if len(r['question'].encode()) <= 512 and len(r['code'].encode()) < 512]
train = [r for r in short_rows if not is_validation(r['question'])][:512]
validation = [r for r in short_rows if is_validation(r['question'])][:64]
assert train and validation
assert not ({r['question'] for r in train} & {r['question'] for r in validation})
test_questions = pq.read_table(cache / 'test.parquet')['question'].to_pylist()
assert len(test_questions) == 1919
assert not ({r['question'] for r in train} & set(test_questions))
print('Rows:', len(train), 'train,', len(validation), 'validation,', len(test_questions), 'test')



In [ ]:
# 2. Byte encoding: reversible, no tokenizer training or external vocabulary.
def encode(text):
    return [b + 3 for b in text.encode('utf-8')]
def decode(ids):
    return bytes(i - 3 for i in ids if 3 <= i < VOCAB).decode('utf-8', errors='replace')
def source_batch(questions):
    return pad_sequence([torch.tensor([BOS] + encode(q) + [EOS]) for q in questions],
                        batch_first=True, padding_value=PAD).to(device)
def training_batch(examples):
    src = source_batch([r['question'] for r in examples])
    codes = [encode(r['code']) for r in examples]
    tgt = pad_sequence([torch.tensor([BOS] + c) for c in codes], batch_first=True, padding_value=PAD).to(device)
    labels = pad_sequence([torch.tensor(c + [EOS]) for c in codes], batch_first=True, padding_value=PAD).to(device)
    return src, tgt, labels



In [ ]:
# 3. Standard encoder-decoder: 2 + 2 layers, learned positions, ReLU, Post-LN.
class SimpleTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.tokens = nn.Embedding(VOCAB, D_MODEL, padding_idx=PAD)
        self.positions = nn.Embedding(MAX_POSITIONS, D_MODEL)
        self.transformer = nn.Transformer(d_model=D_MODEL, nhead=4,
            num_encoder_layers=N_LAYERS, num_decoder_layers=N_LAYERS,
            dim_feedforward=4 * D_MODEL, dropout=0.1, batch_first=True)
        self.transformer.encoder.enable_nested_tensor = False
        self.transformer.encoder.use_nested_tensor = False
        self.head = nn.Linear(D_MODEL, VOCAB)
    def embed(self, ids):
        if ids.shape[1] > MAX_POSITIONS:
            raise ValueError('Sequence too long; no silent truncation')
        pos = torch.arange(ids.shape[1], device=ids.device)
        return self.tokens(ids) * math.sqrt(D_MODEL) + self.positions(pos)
    def decode_prefix(self, tgt, memory, source_padding):
        mask = torch.ones(tgt.shape[1], tgt.shape[1], device=tgt.device, dtype=torch.bool).triu(1)
        return self.transformer.decoder(self.embed(tgt), memory, tgt_mask=mask,
            tgt_key_padding_mask=tgt.eq(PAD), memory_key_padding_mask=source_padding)
    def forward(self, src, tgt):
        memory = self.transformer.encoder(self.embed(src), src_key_padding_mask=src.eq(PAD))
        return self.head(self.decode_prefix(tgt, memory, src.eq(PAD)))
    @torch.inference_mode()
    def generate(self, src, max_new_tokens=512):
        self.eval()
        memory = self.transformer.encoder(self.embed(src), src_key_padding_mask=src.eq(PAD))
        tgt = torch.full((len(src), 1), BOS, device=src.device, dtype=torch.long)
        done = torch.zeros(len(src), device=src.device, dtype=torch.bool)
        for _ in range(max_new_tokens):
            hidden = self.decode_prefix(tgt, memory, src.eq(PAD))
            logits = self.head(hidden[:, -1])
            logits[:, [PAD, BOS]] = -torch.inf
            token = logits.argmax(-1).masked_fill(done, PAD)
            done |= token.eq(EOS)
            tgt = torch.cat([tgt, token[:, None]], dim=1)
            if bool(done.all()):
                break
        return tgt[:, 1:]

model = SimpleTransformer().to(device)
criterion = nn.CrossEntropyLoss(ignore_index=PAD)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
print('Parameters:', sum(p.numel() for p in model.parameters()))



In [ ]:
# 4. A short first run. No AMP, cache or long training; log both train/validation loss.
if device.type == 'cuda':
    torch.cuda.synchronize()
started = time.perf_counter()
training_log = []
for step in range(1, STEPS + 1):
    model.train()
    src, tgt, labels = training_batch(random.sample(train, BATCH_SIZE))
    logits = model(src, tgt)
    loss = criterion(logits.reshape(-1, VOCAB), labels.reshape(-1))
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    if step == 1 or step % 50 == 0:
        model.eval()
        with torch.no_grad():
            src, tgt, labels = training_batch(validation[:16])
            val_logits = model(src, tgt)
            val_loss = criterion(val_logits.reshape(-1, VOCAB), labels.reshape(-1)).item()
        entry = {'step': step, 'train_loss': loss.item(), 'validation_loss_16': val_loss}
        training_log.append(entry)
        print('LOSS', json.dumps(entry), flush=True)
if device.type == 'cuda':
    torch.cuda.synchronize()
training_seconds = time.perf_counter() - started
print('Training seconds:', round(training_seconds, 2))
torch.save({'model': {k: v.cpu() for k, v in model.state_dict().items()},
            'steps': STEPS, 'seed': SEED, 'optimizer': optimizer.state_dict()}, 'hw02-first-baseline.pt')



In [ ]:
# 5. Look at generated text. Do not execute generated or reference programs.
examples = []
for row in validation[:3]:
    ids = model.generate(source_batch([row['question']]))[0].cpu().tolist()
    answer = decode(ids)
    try:
        ast.parse(answer)
        parses = True
    except SyntaxError:
        parses = False
    example = {'question': row['question'], 'generated_code': answer, 'parses_as_python': parses}
    examples.append(example)
    print('EXAMPLE', json.dumps(example, ensure_ascii=False), flush=True)



In [ ]:
# 6. Real whole-test greedy generation with a 180-second stopping budget.
# Setup/download/training are not included; tokenization, transfers, decoding and saving are.
model.eval()
if device.type == 'cuda':
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
started = time.perf_counter()
answers, byte_tokens = [], 0
for offset in range(0, len(test_questions), 32):
    if time.perf_counter() - started >= 180:
        break
    generated = model.generate(source_batch(test_questions[offset:offset + 32]))
    if device.type == 'cuda':
        torch.cuda.synchronize()
    for ids in generated.cpu().tolist():
        byte_tokens += sum(i != PAD for i in ids)
        answers.append(decode(ids))
    if len(answers) % 256 == 0:
        print('INFERENCE', len(answers), 'of', len(test_questions), flush=True)
Path('hw02-exploratory-predictions.json').write_text(json.dumps(answers, ensure_ascii=False))
if device.type == 'cuda':
    torch.cuda.synchronize()
seconds = time.perf_counter() - started
result = {'seed': SEED, 'steps': STEPS, 'training_examples': len(train),
    'validation_examples': len(validation), 'training_seconds': training_seconds,
    'parameters': sum(p.numel() for p in model.parameters()),
    'torch_version': torch.__version__,
    'gpu': torch.cuda.get_device_name(0) if device.type == 'cuda' else 'CPU',
    'dtype': str(next(model.parameters()).dtype), 'tokenizer': 'UTF-8 bytes',
    'batch_size_inference': 32, 'max_new_tokens': 512,
    'test_examples': len(test_questions), 'completed': len(answers), 'seconds': seconds,
    'answers_per_second': len(answers) / seconds, 'generated_tokens': byte_tokens,
    'tokens_per_second': byte_tokens / seconds,
    'mean_generated_tokens': byte_tokens / max(len(answers), 1),
    'complete': len(answers) == len(test_questions),
    'within_180_seconds': len(answers) == len(test_questions) and seconds <= 180,
    'peak_gpu_memory_mib': torch.cuda.max_memory_allocated() / 2**20 if device.type == 'cuda' else None,
    'functional_quality_evaluated': False, 'training_log': training_log, 'examples': examples}
Path('hw02-first-results.json').write_text(json.dumps(result, ensure_ascii=False, indent=2))
print('RESULT', json.dumps({k: v for k, v in result.items() if k not in ['training_log', 'examples']}, ensure_ascii=False), flush=True)
print('Exploratory outputs only; the submission format/checker are not established.')
